In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/digit-recognizer/sample_submission.csv
/kaggle/input/competitions/digit-recognizer/train.csv
/kaggle/input/competitions/digit-recognizer/test.csv


In [3]:
sample_path = "/kaggle/input/competitions/digit-recognizer/sample_submission.csv"
train_path = "/kaggle/input/competitions/digit-recognizer/train.csv"
test_path = "/kaggle/input/competitions/digit-recognizer/test.csv"

In [46]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [47]:
df_train = pd.read_csv(train_path)
df_test = pd.read_csv(test_path)

In [48]:
def data_analysis(df):
    print(df.isnull().sum())


In [49]:
data_analysis(df_train)
data_analysis(df_test)

# data is clean not need to any preprocessing

label       0
pixel0      0
pixel1      0
pixel2      0
pixel3      0
           ..
pixel779    0
pixel780    0
pixel781    0
pixel782    0
pixel783    0
Length: 785, dtype: int64
pixel0      0
pixel1      0
pixel2      0
pixel3      0
pixel4      0
           ..
pixel779    0
pixel780    0
pixel781    0
pixel782    0
pixel783    0
Length: 784, dtype: int64


In [50]:
X = df_train.drop(columns=['label'])
y = df_train["label"]
print(X.shape)
print(y.shape)

(42000, 784)
(42000,)


In [51]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42, stratify=y)


In [52]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [53]:
import torch 

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cpu


In [54]:
import torch
from torch.utils.data import TensorDataset, DataLoader

# Convert to tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train.to_numpy(), dtype=torch.float32).reshape(-1, 1)

X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test.to_numpy(), dtype=torch.float32).reshape(-1, 1)

# TensorDataset
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

# DataLoader
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [55]:
import torch
import torch.nn as nn

class ImageNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(784, 128)
        self.relu1 = nn.ReLU()
        self.layer2 = nn.Linear(128, 64)
        self.relu2 = nn.ReLU()
        self.layer3 = nn.Linear(64, 32)
        self.relu3 = nn.ReLU()
        self.out = nn.Linear(32, 10)

    def forward(self, x):
        x = self.relu1(self.layer1(x))
        x = self.relu2(self.layer2(x))
        x = self.relu3(self.layer3(x))
        return self.out(x)   # raw logits — softmax nahi, jaisa bola tha


In [61]:
import torch
import torch.nn as nn

class ImageNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(784, 128)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(0.2)
        self.layer2 = nn.Linear(128, 64)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(0.2)
        self.layer3 = nn.Linear(64, 32)
        self.relu3 = nn.ReLU()
        self.out = nn.Linear(32, 10)

    def forward(self, x):
        x = self.dropout1(self.relu1(self.layer1(x)))
        x = self.dropout2(self.relu2(self.layer2(x)))
        x = self.relu3(self.layer3(x))
        return self.out(x)   # raw logits


In [62]:
model = ImageNet().to(device)

In [63]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

loss_history = []

epochs = 20

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for x_batch, y_batch in train_loader:
        optimizer.zero_grad()

        logits = model(x_batch)
        y_batch = y_batch.view(-1).long()
        loss = criterion(logits, y_batch)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        pred = logits.argmax(dim=1)
        correct += (pred == y_batch).sum().item()
        total += y_batch.size(0)

    epoch_loss = running_loss / len(train_loader)
    epoch_acc = correct / total
    loss_history.append(epoch_loss)

    print(f"Epoch {epoch+1}/{epochs} — Loss: {epoch_loss:.4f} — Acc: {epoch_acc:.4f}")


Epoch 1/20 — Loss: 0.4010 — Acc: 0.8769
Epoch 2/20 — Loss: 0.1968 — Acc: 0.9418
Epoch 3/20 — Loss: 0.1655 — Acc: 0.9517
Epoch 4/20 — Loss: 0.1339 — Acc: 0.9582
Epoch 5/20 — Loss: 0.1202 — Acc: 0.9624
Epoch 6/20 — Loss: 0.1101 — Acc: 0.9657
Epoch 7/20 — Loss: 0.1032 — Acc: 0.9686
Epoch 8/20 — Loss: 0.0957 — Acc: 0.9694
Epoch 9/20 — Loss: 0.0872 — Acc: 0.9726
Epoch 10/20 — Loss: 0.0829 — Acc: 0.9741
Epoch 11/20 — Loss: 0.0769 — Acc: 0.9757
Epoch 12/20 — Loss: 0.0771 — Acc: 0.9757
Epoch 13/20 — Loss: 0.0722 — Acc: 0.9777
Epoch 14/20 — Loss: 0.0675 — Acc: 0.9785
Epoch 15/20 — Loss: 0.0655 — Acc: 0.9788
Epoch 16/20 — Loss: 0.0656 — Acc: 0.9797
Epoch 17/20 — Loss: 0.0591 — Acc: 0.9816
Epoch 18/20 — Loss: 0.0658 — Acc: 0.9803
Epoch 19/20 — Loss: 0.0626 — Acc: 0.9811
Epoch 20/20 — Loss: 0.0525 — Acc: 0.9833


In [64]:
model.eval()
all_preds, all_true = [], []
total_loss = 0.0

with torch.no_grad():
    for x_batch, y_batch in test_loader:
        y_batch = y_batch.view(-1).long()
        logits = model(x_batch)

        loss = criterion(logits, y_batch)   # ye line add karni thi
        total_loss += loss.item()

        pred = logits.argmax(dim=1)
        all_preds.extend(pred.cpu().numpy())
        all_true.extend(y_batch.cpu().numpy())

test_loss = total_loss / len(test_loader)
print("Test Loss:", test_loss)


Test Loss: 0.16359308139012874


In [65]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

acc = accuracy_score(all_true, all_preds)
print("Test Accuracy:", acc)

print(classification_report(all_true, all_preds))
print(confusion_matrix(all_true, all_preds))


Test Accuracy: 0.9675
              precision    recall  f1-score   support

           0       0.98      0.97      0.97       827
           1       0.98      0.99      0.99       937
           2       0.96      0.98      0.97       835
           3       0.96      0.96      0.96       870
           4       0.97      0.97      0.97       814
           5       0.97      0.93      0.95       759
           6       0.98      0.98      0.98       827
           7       0.97      0.97      0.97       880
           8       0.97      0.94      0.96       813
           9       0.94      0.97      0.95       838

    accuracy                           0.97      8400
   macro avg       0.97      0.97      0.97      8400
weighted avg       0.97      0.97      0.97      8400

[[803   0   4   1   2   2   6   4   3   2]
 [  0 930   5   1   0   0   0   0   1   0]
 [  2   2 818   4   1   0   2   4   1   1]
 [  2   0  11 836   1   9   0   4   2   5]
 [  0   3   5   0 787   0   3   0   0  16]
 [  

In [66]:
import torch
import torch.nn as nn

class ImageNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(784, 256)
        self.bn1 = nn.BatchNorm1d(256)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(0.3)
        self.layer2 = nn.Linear(256, 128)
        self.bn2 = nn.BatchNorm1d(128)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(0.3)
        self.out = nn.Linear(128, 10)

    def forward(self, x):
        x = self.dropout1(self.relu1(self.bn1(self.layer1(x))))
        x = self.dropout2(self.relu2(self.bn2(self.layer2(x))))
        return self.out(x)   # raw logits


In [67]:
model = ImageNet().to(device)

In [68]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

loss_history = []

epochs = 20

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for x_batch, y_batch in train_loader:
        optimizer.zero_grad()

        logits = model(x_batch)
        y_batch = y_batch.view(-1).long()
        loss = criterion(logits, y_batch)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        pred = logits.argmax(dim=1)
        correct += (pred == y_batch).sum().item()
        total += y_batch.size(0)

    epoch_loss = running_loss / len(train_loader)
    epoch_acc = correct / total
    loss_history.append(epoch_loss)

    print(f"Epoch {epoch+1}/{epochs} — Loss: {epoch_loss:.4f} — Acc: {epoch_acc:.4f}")


Epoch 1/20 — Loss: 0.3970 — Acc: 0.8868
Epoch 2/20 — Loss: 0.2139 — Acc: 0.9337
Epoch 3/20 — Loss: 0.1781 — Acc: 0.9440
Epoch 4/20 — Loss: 0.1490 — Acc: 0.9535
Epoch 5/20 — Loss: 0.1328 — Acc: 0.9581
Epoch 6/20 — Loss: 0.1197 — Acc: 0.9622
Epoch 7/20 — Loss: 0.1092 — Acc: 0.9646
Epoch 8/20 — Loss: 0.1055 — Acc: 0.9657
Epoch 9/20 — Loss: 0.0941 — Acc: 0.9688
Epoch 10/20 — Loss: 0.0879 — Acc: 0.9717
Epoch 11/20 — Loss: 0.0827 — Acc: 0.9725
Epoch 12/20 — Loss: 0.0807 — Acc: 0.9739
Epoch 13/20 — Loss: 0.0762 — Acc: 0.9751
Epoch 14/20 — Loss: 0.0744 — Acc: 0.9765
Epoch 15/20 — Loss: 0.0695 — Acc: 0.9770
Epoch 16/20 — Loss: 0.0623 — Acc: 0.9788
Epoch 17/20 — Loss: 0.0620 — Acc: 0.9796
Epoch 18/20 — Loss: 0.0620 — Acc: 0.9793
Epoch 19/20 — Loss: 0.0599 — Acc: 0.9803
Epoch 20/20 — Loss: 0.0567 — Acc: 0.9817


In [69]:
model.eval()
all_preds, all_true = [], []
total_loss = 0.0

with torch.no_grad():
    for x_batch, y_batch in test_loader:
        y_batch = y_batch.view(-1).long()
        logits = model(x_batch)

        loss = criterion(logits, y_batch)   # ye line add karni thi
        total_loss += loss.item()

        pred = logits.argmax(dim=1)
        all_preds.extend(pred.cpu().numpy())
        all_true.extend(y_batch.cpu().numpy())

test_loss = total_loss / len(test_loader)
print("Test Loss:", test_loss)



Test Loss: 0.0960633207436566


In [71]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

acc = accuracy_score(all_true, all_preds)
print("Test Accuracy:", acc)

print(classification_report(all_true, all_preds))
print(confusion_matrix(all_true, all_preds))


Test Accuracy: 0.9764285714285714
              precision    recall  f1-score   support

           0       0.99      0.99      0.99       827
           1       0.99      0.99      0.99       937
           2       0.98      0.98      0.98       835
           3       0.97      0.96      0.97       870
           4       0.96      0.99      0.97       814
           5       0.97      0.96      0.97       759
           6       0.98      1.00      0.99       827
           7       0.97      0.98      0.98       880
           8       0.98      0.97      0.97       813
           9       0.97      0.95      0.96       838

    accuracy                           0.98      8400
   macro avg       0.98      0.98      0.98      8400
weighted avg       0.98      0.98      0.98      8400

[[817   0   0   1   3   1   4   0   1   0]
 [  0 932   2   1   0   0   0   0   2   0]
 [  1   1 817   3   4   0   0   6   3   0]
 [  1   1   8 838   0  10   0   5   1   6]
 [  0   2   0   0 804   2   2   0  

In [87]:
from torch.utils.data import TensorDataset, DataLoader

X_new = scaler.transform(df_test)
X_new_tensor = torch.tensor(X_new, dtype=torch.float32)

test_dataset = TensorDataset(X_new_tensor)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)


model.eval()
preds = []


with torch.no_grad():
    for (x_batch,) in test_loader:
        logits = model(x_batch)
        batch_pred = logits.argmax(dim=1)   # har image ka sirf 1 digit
        preds.extend(batch_pred.cpu().numpy())

print(len(preds))   # ab 28000 aana chahiye

        
submission = pd.DataFrame({
    'ImageId': df_test.index + 1,
    'Label': preds
})
submission.to_csv('submission.csv', index=False)




28000


In [85]:
len(preds)

280000

In [86]:
df_test.shape

(28000, 784)